In [1]:
# === Setup: load both attacking data sources for comparison ===
import pandas as pd
import numpy as np

BASE = r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot"

# Source A: Understat season aggregates (what we currently use — 10 seasons, stable id)
us = pd.read_parquet(BASE + r"\data\history\understat_season_aggregates.parquet")
for c in ["time","goals","xG","assists","xA","shots","npg","npxG"]:
    us[c] = pd.to_numeric(us[c])

# Source B: Core-Insights matchstats (per-match, richer, but only 2 seasons)
ms = pd.read_parquet(BASE + r"\data\history\core_insights_matchstats.parquet")
ms = ms[ms["match_id"].str.contains("-prem-", na=False)].copy()   # PL only

print("=== Understat season aggregates (Source A) ===")
print("Seasons:", sorted(us["understat_season"].unique()))
print("Attacking cols: xG, xA, npxG, shots (season totals)\n")

print("=== Core-Insights matchstats (Source B) ===")
print("Seasons:", sorted(ms["season"].unique()))
print("Per-match attacking cols present:",
      [c for c in ["xg","xa","xgot","shots_on_target","big_chances_missed",
                   "total_shots","chances_created","touches_opposition_box"] if c in ms.columns])
print("Rows:", len(ms))

=== Understat season aggregates (Source A) ===
Seasons: ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Attacking cols: xG, xA, npxG, shots (season totals)

=== Core-Insights matchstats (Source B) ===
Seasons: ['2024-2025', '2025-2026']
Per-match attacking cols present: ['xg', 'xa', 'xgot', 'shots_on_target', 'big_chances_missed', 'total_shots', 'chances_created', 'touches_opposition_box']
Rows: 24028


In [2]:
# === Are Source B's richer per-match signals more persistent than plain xG? ===
# Attach position + gw. Aggregate per player-season-half. Test which signal best
# predicts next-period GOALS (the thing we ultimately care about).

gwref = pd.read_parquet(BASE + r"\data\history\core_insights_gameweek_stats.parquet")
pos_map = gwref[["id","position"]].drop_duplicates("id").set_index("id")["position"]
ms["position"] = ms["player_id"].map(pos_map)

signals = ["xg","xa","xgot","shots_on_target","total_shots",
           "chances_created","touches_opposition_box","big_chances_missed"]
for c in signals + ["goals","minutes_played"]:
    ms[c] = pd.to_numeric(ms[c], errors="coerce")

# Split each season into halves, compute per-90 rates, test half1 signal -> half2 GOALS
def half_predict(position, min_mins=450):
    rows=[]
    for season in ["2024-2025","2025-2026"]:
        s = ms[(ms["season"]==season) & (ms["position"]==position)].copy()
        med = s["gw"].median()
        h1 = s[s["gw"]<=med]; h2 = s[s["gw"]>med]
        def agg(h):
            g = h.groupby("player_id").agg(mins=("minutes_played","sum"),
                    **{c:(c,"sum") for c in signals+["goals"]}).reset_index()
            g = g[g["mins"]>=min_mins]
            for c in signals+["goals"]:
                g[c+"_90"]=g[c]/g["mins"]*90
            return g
        a, b = agg(h1), agg(h2)
        m = a.merge(b[["player_id","goals_90"]], on="player_id", suffixes=("","_next"))
        rows.append(m)
    return pd.concat(rows)

print("Which signal (half 1) best predicts next-half GOALS/90? (correlation)\n")
for position in ["Forward","Midfielder"]:
    d2 = half_predict(position)
    print(f"{position} (n={len(d2)}):")
    cors = {sig: d2[sig+"_90"].corr(d2["goals_90_next"]) for sig in signals}
    for sig, c in sorted(cors.items(), key=lambda x:-abs(x[1])):
        print(f"    {sig:24s} {c:+.3f}")
    print()

Which signal (half 1) best predicts next-half GOALS/90? (correlation)

Forward (n=53):
    big_chances_missed       +0.597
    xg                       +0.559
    touches_opposition_box   +0.548
    total_shots              +0.530
    xgot                     +0.471
    shots_on_target          +0.469
    xa                       -0.097
    chances_created          -0.047

Midfielder (n=236):
    xg                       +0.642
    total_shots              +0.622
    shots_on_target          +0.611
    xgot                     +0.574
    touches_opposition_box   +0.549
    big_chances_missed       +0.534
    chances_created          +0.284
    xa                       +0.251



In [3]:
# === Does RECENT per-match form predict next-match goals? (with vs without xG) ===
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Build per-match rolling features (shift-then-roll, within season), predict next-match goals/90
ms_s = ms[ms["position"].isin(["Forward","Midfielder"])].copy()
ms_s = ms_s.sort_values(["player_id","season","gw"])
ms_s["mins"] = ms_s["minutes_played"].clip(lower=1)
for c in ["xg","xgot","total_shots","shots_on_target","touches_opposition_box","chances_created","goals"]:
    ms_s[c+"_p90"] = ms_s[c]/ms_s["mins"]*90

grp = ms_s.groupby(["season","player_id"])
def rollf(col, w): return grp[col].transform(lambda s: s.shift(1).rolling(w, min_periods=2).mean())

# recent-form features (last 5 matches)
ms_s["xg_r5"]      = rollf("xg_p90",5)
ms_s["xgot_r5"]    = rollf("xgot_p90",5)
ms_s["shots_r5"]   = rollf("total_shots_p90",5)
ms_s["sot_r5"]     = rollf("shots_on_target_p90",5)
ms_s["box_r5"]     = rollf("touches_opposition_box_p90",5)
ms_s["cc_r5"]      = rollf("chances_created_p90",5)
# season-average-so-far (expanding) xg = the "baseline" that season aggregates approximate
ms_s["xg_expand"]  = grp["xg_p90"].transform(lambda s: s.shift(1).expanding(min_periods=3).mean())
# target: this match's goals/90
ms_s["target"] = ms_s["goals_p90"]

d3 = ms_s.dropna(subset=["xg_r5","xg_expand","xgot_r5","shots_r5","sot_r5","box_r5","cc_r5","target"]).copy()
d3 = d3[d3["mins"]>=45]   # meaningful appearances
print("Match rows for the test:", len(d3))

def cv_rmse(X, y):
    m = LinearRegression().fit(X, y)
    return np.sqrt(mean_squared_error(y, m.predict(X)))

y = d3["target"]
print("\nPredicting next-match goals/90 — RMSE (lower better):\n")
# 1. baseline: season-average xG only
print(f"  season-avg xG only          : {cv_rmse(d3[['xg_expand']], y):.4f}")
# 2. recent-form xG only
print(f"  recent-5 xG only            : {cv_rmse(d3[['xg_r5']], y):.4f}")
# 3. WITH xG: season-avg + recent xG + recent other signals
with_xg = ["xg_expand","xg_r5","xgot_r5","shots_r5","sot_r5","box_r5","cc_r5"]
print(f"  WITH xG (season+recent+rich): {cv_rmse(d3[with_xg], y):.4f}")
# 4. WITHOUT xG: only non-xG recent signals
without_xg = ["shots_r5","sot_r5","box_r5","cc_r5"]
print(f"  WITHOUT xG (non-xG signals) : {cv_rmse(d3[without_xg], y):.4f}")

Match rows for the test: 8381

Predicting next-match goals/90 — RMSE (lower better):

  season-avg xG only          : 0.4139
  recent-5 xG only            : 0.4189
  WITH xG (season+recent+rich): 0.4129
  WITHOUT xG (non-xG signals) : 0.4172


In [5]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

with_xg = ["xg_expand","xg_r5","xgot_r5","shots_r5","sot_r5","box_r5","cc_r5"]

tr = d3[d3["season"]=="2024-2025"]
te = d3[d3["season"]=="2025-2026"]
Xtr, ytr = tr[with_xg], tr["target"]
Xte, yte = te[with_xg], te["target"]
print(f"Train {len(tr)}, test {len(te)}\n")

base_rmse = np.sqrt(mean_squared_error(yte, te["xg_expand"]))
print(f"  {'baseline (xG as prediction)':32s}: {base_rmse:.4f}")

sc = StandardScaler().fit(Xtr)
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=200, num_leaves=15, learning_rate=0.05, min_child_samples=40, random_state=42, verbose=-1),
    "SVR": SVR(C=1.0),
}
for name, m in models.items():
    if name=="SVR":
        m.fit(sc.transform(Xtr), ytr); p = m.predict(sc.transform(Xte))
    else:
        m.fit(Xtr, ytr); p = m.predict(Xte)
    print(f"  {name:32s}: {np.sqrt(mean_squared_error(yte, p)):.4f}")

Train 4428, test 3953

  baseline (xG as prediction)     : 0.5310
  LinearRegression                : 0.4790
  RandomForest                    : 0.4648
  XGBoost                         : 0.4808
  LightGBM                        : 0.4739
  SVR                             : 0.4803


In [6]:
# === Can we bridge Understat id <-> Core-Insights player_id? (needed to blend the two sources) ===
# The crosswalk maps element <-> understat_id (2025-26). Core-Insights player_id... let's check.
cw = pd.read_csv(BASE + r"\data\history\player_id_crosswalk_final.csv")
print("Crosswalk columns:", cw.columns.tolist())

# Core-Insights gw file: does its 'id' == crosswalk 'player_id' == crosswalk 'element'?
# We saw element==player_id in crosswalk. Core-Insights uses 'id' in gw file.
gw_ids = set(gwref["id"].unique())
cw_pid = set(cw["player_id"].unique())
cw_el  = set(cw["element"].unique())
print(f"\nCore-Insights gw 'id' overlap with crosswalk player_id: {len(gw_ids & cw_pid)}")
print(f"Core-Insights gw 'id' overlap with crosswalk element:    {len(gw_ids & cw_el)}")

# And crosswalk understat_id links to Understat 'id'
us_ids = set(us["id"].astype(str).unique())
cw_us  = set(cw["understat_id"].dropna().astype(int).astype(str).unique())
print(f"Understat 'id' overlap with crosswalk understat_id:      {len(us_ids & cw_us)}")

# So the bridge (2025-26 only): Core-Insights id -> element -> understat_id -> Understat id
print("\nBridge viability: Core-Insights id == element (crosswalk), then element -> understat_id")
print("This works for 2025-26 only (crosswalk is single-season).")

Crosswalk columns: ['element', 'player_id', 'understat_id', 'matched_name']

Core-Insights gw 'id' overlap with crosswalk player_id: 841
Core-Insights gw 'id' overlap with crosswalk element:    841
Understat 'id' overlap with crosswalk understat_id:      525

Bridge viability: Core-Insights id == element (crosswalk), then element -> understat_id
This works for 2025-26 only (crosswalk is single-season).
